In [1]:
import os
import time
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import ta



In [2]:
# ==========================================================
# Module 1 : Configuration
# Statistical Trading Engine
# ==========================================================

import os

# ==========================================================
# File Paths
# ==========================================================

# Daily Scanner Output
BULLISH_SHARES_FILE = (
    "/home/devinderjeet/report/Bullish_Shares.xlsx"
)

# 5 Minute Data Folder
DATA_FOLDER = (
    "/home/hadoop/shareMarket_Data"
)

# Output Folder
OUTPUT_FOLDER = (
    "/home/devinderjeet/report"
)

# Log File
LOG_FILE = (
    os.path.join(
        OUTPUT_FOLDER,
        "StatisticalEngine.log"
    )
)

# ==========================================================
# Scheduler
# ==========================================================

# Scan every 5 minutes
REFRESH_INTERVAL = 300

# Wait before first scan
INITIAL_DELAY = 5

# ==========================================================
# Statistical Settings
# ==========================================================

# Number of latest candles used
LOOKBACK_BARS = 100

# Minimum bars required
MINIMUM_BARS = 50

# ==========================================================
# Z Score
# ==========================================================

# Strong Buy
ZSCORE_STRONG_BUY = -2.0

# Buy
ZSCORE_BUY = -1.5

# Neutral
ZSCORE_WAIT = 0.5

# Sell
ZSCORE_SELL = 1.5

# ==========================================================
# Probability
# ==========================================================

MINIMUM_PROBABILITY = 80

# ==========================================================
# Entry Calculation
# ==========================================================

ENTRY_STD_MULTIPLIER = 0.25

# Entry Price

# Mean - (Std × Multiplier)

# ==========================================================
# Risk Management
# ==========================================================

ATR_STOPLOSS_MULTIPLIER = 1.5

ATR_TARGET_MULTIPLIER = 3.0

MINIMUM_RISK_REWARD = 2.0

# ==========================================================
# Daily Filters
# ==========================================================

REQUIRED_DAILY_SIGNAL = "BUY"

MINIMUM_DAILY_CONFIDENCE = 60

# ==========================================================
# Volume
# ==========================================================

USE_VOLUME_CONFIRMATION = True

VOLUME_MULTIPLIER = 1.20

# ==========================================================
# Telegram
# ==========================================================

ENABLE_TELEGRAM = True

# ==========================================================
# Console
# ==========================================================

PRINT_WAIT_STATUS = True

PRINT_BUY_STATUS = True

PRINT_EXIT_STATUS = True

# ==========================================================
# Trade Status
# ==========================================================

STATUS_WAIT = "WAIT"

STATUS_READY = "READY"

STATUS_BUY = "BUY"

STATUS_ENTERED = "ENTERED"

STATUS_TARGET = "TARGET HIT"

STATUS_STOPLOSS = "STOPLOSS HIT"

STATUS_EXIT = "EXIT"

# ==========================================================
# Colors
# ==========================================================

BUY_COLOR = "GREEN"

SELL_COLOR = "RED"

WAIT_COLOR = "YELLOW"

# ==========================================================
# End Configuration
# ==========================================================

print("=" * 80)
print("Statistical Engine Configuration Loaded")
print("=" * 80)
print(f"Data Folder          : {DATA_FOLDER}")
print(f"Bullish Shares File  : {BULLISH_SHARES_FILE}")
print(f"Refresh Interval     : {REFRESH_INTERVAL} Seconds")
print(f"Lookback Bars        : {LOOKBACK_BARS}")
print(f"Minimum Probability  : {MINIMUM_PROBABILITY}%")
print("=" * 80)

Statistical Engine Configuration Loaded
Data Folder          : /home/hadoop/shareMarket_Data
Bullish Shares File  : /home/devinderjeet/report/Bullish_Shares.xlsx
Refresh Interval     : 300 Seconds
Lookback Bars        : 100
Minimum Probability  : 80%


In [3]:
# ==========================================================
# Module 2 : DataLoader
# ==========================================================

import os
import time
import pandas as pd


class DataLoader:

    # ======================================================
    # Constructor
    # ======================================================

    def __init__(self, report_file, data_folder):

        self.report_file = report_file
        self.data_folder = data_folder

        self.columns = [
            "Symbol",
            "Date",
            "Time",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]

        print("Statistical DataLoader Initialized")

    # ======================================================
    # Load Bullish Shares
    # ======================================================

    def load_bullish_shares(self):

        if not os.path.exists(self.report_file):

            print("Bullish report not found.")

            return pd.DataFrame()

        df = pd.read_excel(self.report_file)

        if len(df) == 0:

            return pd.DataFrame()

        if "Symbol" not in df.columns:

            raise ValueError(
                "'Symbol' column not found."
            )

        df["Symbol"] = (

            df["Symbol"]

            .astype(str)

            .str.upper()

            .str.strip()

        )

        return df

    # ======================================================
    # Load One 5 Minute File
    # ======================================================

    def load_5min_data(self, symbol):

        filename = os.path.join(

            self.data_folder,

            f"{symbol}_5mins.txt"

        )

        if not os.path.exists(filename):

            print(f"{symbol} file not found.")

            return None

        df = pd.read_csv(

            filename,

            header=None,

            names=self.columns

        )

        # Remove spaces
        df.columns = [c.strip() for c in df.columns]

        # Numeric conversion
        numeric = [

            "Open",
            "High",
            "Low",
            "Close",
            "Volume"

        ]

        for col in numeric:

            df[col] = pd.to_numeric(

                df[col],

                errors="coerce"

            )

        df.dropna(inplace=True)

        # Create Datetime
        df["Datetime"] = pd.to_datetime(

            df["Date"].astype(str)

            + " "

            + df["Time"].astype(str),

            errors="coerce"

        )

        df.dropna(subset=["Datetime"], inplace=True)

        df.sort_values(

            "Datetime",

            inplace=True

        )

        df.reset_index(

            drop=True,

            inplace=True

        )

        return df

    # ======================================================
    # Load All Shares
    # ======================================================

    def load_all(self):

        bullish_df = self.load_bullish_shares()

        all_data = {}

        if bullish_df.empty:

            return all_data

        for symbol in bullish_df["Symbol"]:

            df = self.load_5min_data(symbol)

            if df is not None:

                all_data[symbol] = df

        return all_data

    # ======================================================
    # Refresh
    # ======================================================

    def refresh(self, interval=300):

        while True:

            print("="*80)
            print("Refreshing 5 Minute Data")
            print("="*80)

            yield self.load_all()

            time.sleep(interval)

In [4]:
loader = DataLoader(

    BULLISH_SHARES_FILE,

    DATA_FOLDER

)

symbols = loader.load_bullish_shares()

print(symbols)

Statistical DataLoader Initialized
       Symbol   Close  Final Score  Confidence Signal    Trend  Trend Score  \
0        SBIN   835.4           82          88    BUY  Bullish           26   
1  HINDUNILVR  2648.5           91          94    BUY  Bullish           30   

        Pattern  Support  Resistance   EMA20   EMA50  EMA200   RSI  MACD   ADX  
0     Bull Flag      810         860   820.5   805.6   760.3  63.5  4.21  31.4  
1  Cup & Handle     2580        2720  2615.3  2582.1  2440.8  67.2  8.12  29.7  


In [5]:
# ==========================================================
# Module 3 : Statistical Engine
# ==========================================================

import numpy as np
import pandas as pd


class StatisticalEngine:

    # ======================================================
    # Constructor
    # ======================================================

    def __init__(self):

        print("Statistical Engine Initialized")


    # ======================================================
    # Validate Data
    # ======================================================

    def validate(self, df):

        if df is None:
            raise ValueError("DataFrame is None")

        if len(df) < MINIMUM_BARS:
            raise ValueError(
                f"Minimum {MINIMUM_BARS} candles required."
            )

        required = [
            "Close",
            "High",
            "Low",
            "Volume"
        ]

        for col in required:

            if col not in df.columns:

                raise ValueError(
                    f"{col} column missing."
                )


    # ======================================================
    # Mean
    # ======================================================

    def calculate_mean(
        self,
        series
    ):

        return float(series.mean())


    # ======================================================
    # Median
    # ======================================================

    def calculate_median(
        self,
        series
    ):

        return float(series.median())


    # ======================================================
    # Standard Deviation
    # ======================================================

    def calculate_std(
        self,
        series
    ):

        return float(series.std())


    # ======================================================
    # Variance
    # ======================================================

    def calculate_variance(
        self,
        series
    ):

        return float(series.var())


    # ======================================================
    # Z Score
    # ======================================================

    def calculate_zscore(
        self,
        current_price,
        mean,
        std
    ):

        if std == 0:

            return 0

        return float(

            (current_price - mean)

            / std

        )


    # ======================================================
    # Historical Probability
    # ======================================================

    def calculate_probability(
        self,
        zscore
    ):

        z = abs(zscore)

        if z >= 3:

            return 99

        elif z >= 2.5:

            return 97

        elif z >= 2:

            return 95

        elif z >= 1.5:

            return 88

        elif z >= 1:

            return 75

        elif z >= 0.5:

            return 60

        else:

            return 50


    # ======================================================
    # Main Calculation
    # ======================================================

    def calculate(
        self,
        df
    ):

        self.validate(df)

        data = df.tail(
            LOOKBACK_BARS
        )

        close = data["Close"]

        current_price = float(
            close.iloc[-1]
        )

        mean = self.calculate_mean(close)

        median = self.calculate_median(close)

        std = self.calculate_std(close)

        variance = self.calculate_variance(close)

        zscore = self.calculate_zscore(

            current_price,

            mean,

            std

        )

        probability = self.calculate_probability(
            zscore
        )

        return {

            "Current Price": round(
                current_price,
                2
            ),

            "Mean": round(
                mean,
                2
            ),

            "Median": round(
                median,
                2
            ),

            "Std": round(
                std,
                4
            ),

            "Variance": round(
                variance,
                4
            ),

            "Z Score": round(
                zscore,
                2
            ),

            "Probability": probability

        }

In [6]:
# ==========================================================
# Test Statistical Engine
# ==========================================================

loader = DataLoader(
    BULLISH_SHARES_FILE,
    DATA_FOLDER
)

engine = StatisticalEngine()

shares = loader.load_bullish_shares()

symbol = shares.iloc[1]["Symbol"]

print(symbol)

df = loader.load_5min_data(symbol)

result = engine.calculate(df)

print("=" * 60)

for key, value in result.items():

    print(f"{key:<20}: {value}")

Statistical DataLoader Initialized
Statistical Engine Initialized
HINDUNILVR
Current Price       : 2138.0
Mean                : 2138.42
Median              : 2137.95
Std                 : 3.729
Variance            : 13.9053
Z Score             : -0.11
Probability         : 50


In [7]:
shares = loader.load_bullish_shares()

for _, row in shares.iterrows():

    symbol = row["Symbol"]

    daily_signal = row["Signal"]

    confidence = row["Confidence"]

    print(symbol)

SBIN
HINDUNILVR


In [8]:
# ==========================================================
# Module 4 : Entry Engine
# Pure Statistical Entry Engine
# ==========================================================

class EntryEngine:

    # ======================================================
    # Constructor
    # ======================================================

    def __init__(

        self,

        entry_std_multiplier=0.25,

        stoploss_std_multiplier=2.0,

        target_std_multiplier=4.0,

        minimum_probability=80,

        minimum_rr=2.0

    ):

        self.entry_std_multiplier = entry_std_multiplier

        self.stoploss_std_multiplier = stoploss_std_multiplier

        self.target_std_multiplier = target_std_multiplier

        self.minimum_probability = minimum_probability

        self.minimum_rr = minimum_rr

        print("Entry Engine Initialized")

    # ======================================================
    # Calculate Trade
    # ======================================================

    def calculate(

        self,

        statistical_result,

        current_price

    ):

        mean = statistical_result["Mean"]

        std = statistical_result["Std"]

        zscore = statistical_result["Z Score"]

        probability = statistical_result["Probability"]

        # --------------------------------------------------
        # Suggested Entry
        # --------------------------------------------------

        entry = mean - (

            std * self.entry_std_multiplier

        )

        # --------------------------------------------------
        # Stop Loss
        # --------------------------------------------------

        stop_loss = entry - (

            std * self.stoploss_std_multiplier

        )

        # --------------------------------------------------
        # Target
        # --------------------------------------------------

        target = entry + (

            std * self.target_std_multiplier

        )

        # --------------------------------------------------
        # Risk Reward
        # --------------------------------------------------

        risk = entry - stop_loss

        reward = target - entry

        rr = reward / risk

        # --------------------------------------------------
        # Status
        # --------------------------------------------------

        if (

            current_price <= entry

            and probability >= self.minimum_probability

            and rr >= self.minimum_rr

            and zscore <= -2

        ):

            status = STATUS_BUY

        elif (

            current_price <= entry * 1.003

            and probability >= self.minimum_probability

        ):

            status = STATUS_READY

        else:

            status = STATUS_WAIT

        # --------------------------------------------------
        # Return
        # --------------------------------------------------

        return {

            "Current Price": round(current_price,2),

            "Mean": round(mean,2),

            "Std": round(std,2),

            "Z Score": round(zscore,2),

            "Suggested Entry": round(entry,2),

            "Stop Loss": round(stop_loss,2),

            "Target": round(target,2),

            "Risk": round(risk,2),

            "Reward": round(reward,2),

            "Risk Reward": round(rr,2),

            "Probability": probability,

            "Status": status

        }

In [9]:
loader = DataLoader(

    BULLISH_SHARES_FILE,

    DATA_FOLDER

)

stat_engine = StatisticalEngine()

entry_engine = EntryEngine()

shares = loader.load_bullish_shares()

for _, row in shares.iterrows():

    symbol = row["Symbol"]

    daily_signal = row["Signal"]

    confidence = row["Confidence"]

    print("=" * 70)
    print(symbol)
    print("=" * 70)

    df = loader.load_5min_data(symbol)

    if df is None or len(df) == 0:
        print("No 5-minute data found.")
        continue

    stat_result = stat_engine.calculate(df)

    current_price = df.iloc[-1]["Close"]

    trade = entry_engine.calculate(

        stat_result,

        current_price

    )

    print(f"Daily Signal : {daily_signal}")
    print(f"Confidence   : {confidence}")

    print("-" * 70)

    for k, v in trade.items():

        print(f"{k:<20}: {v}")

Statistical DataLoader Initialized
Statistical Engine Initialized
Entry Engine Initialized
SBIN
Daily Signal : BUY
Confidence   : 88
----------------------------------------------------------------------
Current Price       : 1026.4
Mean                : 1049.51
Std                 : 9.4
Z Score             : -2.46
Suggested Entry     : 1047.16
Stop Loss           : 1028.36
Target              : 1084.76
Risk                : 18.8
Reward              : 37.6
Risk Reward         : 2.0
Probability         : 95
Status              : READY
HINDUNILVR
Daily Signal : BUY
Confidence   : 94
----------------------------------------------------------------------
Current Price       : 2138.0
Mean                : 2138.42
Std                 : 3.73
Z Score             : -0.11
Suggested Entry     : 2137.49
Stop Loss           : 2130.03
Target              : 2152.4
Risk                : 7.46
Reward              : 14.92
Risk Reward         : 2.0
Probability         : 50
Status              : WAIT


In [10]:
# ==========================================================
# Module 5 : Decision Engine
# ==========================================================

class DecisionEngine:

    # ======================================================
    # Constructor
    # ======================================================

    def __init__(self):

        print("Decision Engine Initialized")

    # ======================================================
    # Decision
    # ======================================================

    def decide(

        self,

        statistical_result,

        trade_result,

        current_price

    ):

        probability = statistical_result["Probability"]

        zscore = statistical_result["Z Score"]

        entry = trade_result["Suggested Entry"]

        stoploss = trade_result["Stop Loss"]

        target = trade_result["Target"]

        status = "WAIT"

        reason = ""

        # ==================================================
        # BUY
        # ==================================================

        if (

            current_price <= entry

            and probability >= MINIMUM_PROBABILITY

            and zscore <= ZSCORE_BUY

        ):

            status = "BUY"

            reason = "Price reached statistical entry zone."

        # ==================================================
        # READY
        # ==================================================

        elif (

            current_price <= entry * 1.003

            and probability >= MINIMUM_PROBABILITY

        ):

            status = "READY"

            reason = "Price approaching entry."

        # ==================================================
        # WAIT
        # ==================================================

        else:

            status = "WAIT"

            reason = "Waiting for better entry."

        # ==================================================
        # Return
        # ==================================================

        return {

            "Decision": status,

            "Reason": reason,

            "Entry": round(entry, 2),

            "Current Price": round(current_price, 2),

            "Stop Loss": round(stoploss, 2),

            "Target": round(target, 2),

            "Probability": probability,

            "Z Score": round(zscore, 2)

        }

In [11]:
# ==========================================================
# Module 6 : Scheduler
# ==========================================================

import time


class Scheduler:

    # ======================================================
    # Constructor
    # ======================================================

    def __init__(

        self,

        loader,

        statistical_engine,

        entry_engine,

        decision_engine,

        refresh_interval=300

    ):

        self.loader = loader

        self.statistical_engine = statistical_engine

        self.entry_engine = entry_engine

        self.decision_engine = decision_engine

        self.refresh_interval = refresh_interval

        print("Scheduler Initialized")

    # ======================================================
    # Process One Share
    # ======================================================
    
    def process_share(self, row):
    
        symbol = row["Symbol"]
    
        # --------------------------------------------
        # Load latest 5 minute data
        # --------------------------------------------
    
        df = self.loader.load_5min_data(symbol)
    
        if df is None:
            return
    
        if len(df) < MINIMUM_BARS:
            return
    
        current_price = df.iloc[-1]["Close"]
    
        # --------------------------------------------
        # Statistical Engine
        # --------------------------------------------
    
        stat_result = self.statistical_engine.calculate(df)
    
        # --------------------------------------------
        # Entry Engine
        # --------------------------------------------
    
        trade = self.entry_engine.calculate(
            stat_result,
            current_price
        )
    
        # --------------------------------------------
        # Decision Engine
        # --------------------------------------------
    
        decision = self.decision_engine.decide(
            stat_result,
            trade,
            current_price
        )
    
        # --------------------------------------------
        # Print Final Dashboard
        # --------------------------------------------
        
        print("=" * 80)
        print(symbol)
        print("=" * 80)
        
        print(f"Decision         : {decision['Decision']}")
        print(f"Reason           : {decision['Reason']}")
        
        print("-" * 80)
        
        print(f"Current Price    : {decision['Current Price']}")
        print(f"Suggested Entry  : {decision['Entry']}")
        print(f"Stop Loss        : {decision['Stop Loss']}")
        print(f"Target           : {decision['Target']}")
        
        print("-" * 80)
        
        print(f"Probability      : {decision['Probability']} %")
        print(f"Z Score          : {decision['Z Score']}")
        print(f"Risk Reward      : {trade['Risk Reward']}")
        print(f"Status           : {trade['Status']}")
    # ======================================================
    # Countdown Timer
    # ======================================================

    def countdown(self):

        remaining = self.refresh_interval

        while remaining >= 0:

            mins = remaining // 60

            secs = remaining % 60

            print(

                f"\rNext Scan In : {mins:02d}:{secs:02d}",

                end="",

                flush=True

            )

            time.sleep(1)

            remaining -= 1

        print()

    # ======================================================
    # Main Loop
    # ======================================================

    def run(self):

        print("=" * 80)
        print("STATISTICAL ENGINE STARTED")
        print("=" * 80)

        while True:

            print("\n")
            print("=" * 80)
            print("Loading Bullish Shares...")
            print("=" * 80)

            try:

                shares = self.loader.load_bullish_shares()

                if shares is None:

                    print("No Bullish Shares Found")

                elif len(shares) == 0:

                    print("Bullish Share List Empty")

                else:

                    print(f"Total Shares : {len(shares)}")

                    for _, row in shares.iterrows():

                        try:

                            self.process_share(row)

                        except Exception as e:

                            print(f"{row['Symbol']} Failed : {e}")

            except Exception as e:

                print(f"Scheduler Error : {e}")

            print("\n")
            print("=" * 80)
            print("SCAN COMPLETED")
            print("=" * 80)

            self.countdown()

In [ ]:
loader = DataLoader(

    BULLISH_SHARES_FILE,

    DATA_FOLDER

)

stat_engine = StatisticalEngine()

entry_engine = EntryEngine()

decision_engine = DecisionEngine()

scheduler = Scheduler(

    loader,

    stat_engine,

    entry_engine,

    decision_engine,

    refresh_interval=300

)

scheduler.run()

Statistical DataLoader Initialized
Statistical Engine Initialized
Entry Engine Initialized
Decision Engine Initialized
Scheduler Initialized
STATISTICAL ENGINE STARTED


Loading Bullish Shares...
Total Shares : 2
SBIN
Decision         : BUY
Reason           : Price reached statistical entry zone.
--------------------------------------------------------------------------------
Current Price    : 1026.4
Suggested Entry  : 1047.16
Stop Loss        : 1028.36
Target           : 1084.76
--------------------------------------------------------------------------------
Probability      : 95 %
Z Score          : -2.46
Risk Reward      : 2.0
Status           : READY
HINDUNILVR
Decision         : WAIT
Reason           : Waiting for better entry.
--------------------------------------------------------------------------------
Current Price    : 2138.0
Suggested Entry  : 2137.49
Stop Loss        : 2130.03
Target           : 2152.4
---------------------------------------------------------------------

KeyboardInterrupt: 